In [1]:
### create sample SQlite database with sample data

import sqlite3
import os
os.makedirs("data/databases", exist_ok=True)

In [2]:
conn = sqlite3.connect("data/databases/sample.db")
cursor = conn.cursor()


In [3]:
cursor.execute('''CREATE TABLE IF NOT EXISTS employees (
                    id INTEGER PRIMARY KEY, name TEXT, department TEXT,salary REAL,role TEXT)''')

In [4]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects (
                    id INTEGER PRIMARY KEY, name TEXT, budget REAL, deadline TEXT, lead_id INTEGER, status TEXT)''')   

In [5]:
employees = [
    (1, 'Alice', 'Engineering', 90000, 'Software Engineer'),
    (2, 'Bob', 'Engineering', 95000, 'Senior Software Engineer'),
    (3, 'Charlie', 'HR', 60000, 'HR Manager'),
    (4, 'David', 'Marketing', 70000, 'Marketing Specialist'),
    (5, 'Eve', 'Engineering', 85000, 'Software Engineer')
]
projects = [
    (1, 'Project Alpha', 100000, '2024-12-31', 2, 'Ongoing'),
    (2, 'Project Beta', 150000, '2025-06-30', 1, 'Planned'),
    (3, 'Project Gamma', 50000, '2024-09-30', 3, 'Completed'),
    (4, 'Project Delta', 200000, '2025-03-31', 4, 'Ongoing'),
    (5, 'Project Epsilon', 120000, '2024-11-30', 5, 'Planned')
]

In [11]:
cursor.executemany(
    'INSERT INTO employees VALUES (?, ?, ?, ?, ?)',
    employees
)

cursor.executemany(
    'INSERT INTO projects VALUES (?, ?, ?, ?, ?, ?)',
    projects
)

conn.commit()

IntegrityError: UNIQUE constraint failed: employees.id

In [12]:
cursor.execute("select * from employees")

In [13]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader


In [15]:
db = SQLDatabase.from_uri("sqlite:///data/databases/sample.db")

print(f"Tables in database: {db.get_usable_table_names()}")
print(f"Schema for employees: {db.get_table_info(['employees'])}")
print(f"Schema for projects: {db.get_table_info(['projects'])}")


Tables in database: ['employees', 'projects']
Schema for employees: 
CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary REAL, 
	role TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary	role
1	Alice	Engineering	90000.0	Software Engineer
2	Bob	Engineering	95000.0	Senior Software Engineer
3	Charlie	HR	60000.0	HR Manager
*/
Schema for projects: 
CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	budget REAL, 
	deadline TEXT, 
	lead_id INTEGER, 
	status TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	budget	deadline	lead_id	status
1	Project Alpha	100000.0	2024-12-31	2	Ongoing
2	Project Beta	150000.0	2025-06-30	1	Planned
3	Project Gamma	50000.0	2024-09-30	3	Completed
*/


In [16]:
from typing import List
from langchain_core.documents import Document

def sql_database_to_documents(db_path:str) -> List[Document]:
    """Convert SQL database tables into a list of Documents."""

    # Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    documents = []
    
    for table_name in tables:
        table_name = table_name[0]  # Extract table name from tuple
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()
        
        # Get column names
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [col[1] for col in cursor.fetchall()]
        
        # Create a document for each row
        for row in rows:
            content = "\n".join([f"{col}: {val}" for col, val in zip(columns, row)])
            documents.append(Document(page_content=content, metadata={"source": f"{table_name}"}))

    conn.close()
    return documents

In [17]:
sql_database_to_documents("data/databases/sample.db")

[Document(metadata={'source': 'employees'}, page_content='id: 1\nname: Alice\ndepartment: Engineering\nsalary: 90000.0\nrole: Software Engineer'),
 Document(metadata={'source': 'employees'}, page_content='id: 2\nname: Bob\ndepartment: Engineering\nsalary: 95000.0\nrole: Senior Software Engineer'),
 Document(metadata={'source': 'employees'}, page_content='id: 3\nname: Charlie\ndepartment: HR\nsalary: 60000.0\nrole: HR Manager'),
 Document(metadata={'source': 'employees'}, page_content='id: 4\nname: David\ndepartment: Marketing\nsalary: 70000.0\nrole: Marketing Specialist'),
 Document(metadata={'source': 'employees'}, page_content='id: 5\nname: Eve\ndepartment: Engineering\nsalary: 85000.0\nrole: Software Engineer'),
 Document(metadata={'source': 'projects'}, page_content='id: 1\nname: Project Alpha\nbudget: 100000.0\ndeadline: 2024-12-31\nlead_id: 2\nstatus: Ongoing'),
 Document(metadata={'source': 'projects'}, page_content='id: 2\nname: Project Beta\nbudget: 150000.0\ndeadline: 2025-06